In [ ]:
# ================================================================
# 📘 config_utility.ipynb - Centralized Utility Library
# ================================================================
# This notebook contains all reusable components of the Performance Tool —
# such as configuration loaders, SQL helpers, data conversion utilities,
# logging, and supporting functions for automation and metadata management.
# #
# Purpose:
# Centralizes reusable functions to ensure modularity, consistency,
# and maintainability across notebooks (e.g., config.ipynb, run.ipynb).
# ================================================================

1. 🔧 Initialization and Imports
Import all required Python libraries and Spark utilities.
These libraries support SQL connections, file operations, and DataFrame handling.

In [ ]:
import os
import re
import uuid
import time
import requests
import concurrent.futures
from datetime import datetime

from typing import List, Tuple, Optional
from collections import defaultdict
from functools import reduce

import pandas as pd

from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.functions import col, lit, udf, explode, when
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, LongType, FloatType, ArrayType

from notebookutils import mssparkutils
from pyspark.sql.utils import AnalysisException

spark = SparkSession.builder.getOrCreate()

StatementMeta(, 8eb609f1-4cac-462d-a95e-6e6e1fd9366c, 3, Finished, Available, Finished)

In [ ]:
schemaMain = StructType([
    StructField("id", StringType(), True),
    StructField("Query", StringType(), True),
    StructField("Start Time", TimestampType(), True),
    StructField("End Time", TimestampType(), True),
    StructField("Execution Time", FloatType(), True),
    StructField("Result", StringType(), True)
])

Fetches workspace and dataset IDs from `metadata.vw_LoadTestDAX`
based on the given Track and Report name.

In [ ]:
def fetch_metadata_config(track: str, report: str, config_jdbc_url: str) -> DataFrame | None:
    """
    Query vw_LoadTestDAX to get workspace/dataset IDs.
    Returns a Spark DataFrame (or None if failure).
    """
    sql = (
        "SELECT DISTINCT sourceWorkspaceId, sourceDatasetId, "
        "targetWorkspaceId, targetDatasetId "
        f"FROM metadata.vw_LoadTestDAX "
        f"WHERE trackName = '{track}' AND reportName = '{report}'"
    )
    try:
        df = readSQLData(sql, config_jdbc_url)
        if isinstance(df, list):
            df = spark.createDataFrame(df)
        return df
    except Exception as ex:
        print(f"[ERROR] fetch_metadata_config failed: {ex}")
        return None

Extracts workspace and dataset IDs (Import Mode & Direct Lake Mode)
from the configuration DataFrame returned by `fetch_metadata_config`.

In [ ]:
def extract_ids_from_config_df(df: DataFrame):
    """
    From config DataFrame, extract four IDs.
    Raises if missing or invalid.
    """
    try:
        row = df.first()
        return (
            row["sourceWorkspaceId"],
            row["sourceDatasetId"],
            row["targetWorkspaceId"],
            row["targetDatasetId"]
        )
    except Exception as ex:
        raise RuntimeError(f"Could not extract IDs: {ex}")

Fetches DAX query definitions (Visual ID, Type, Title, Query Text)
from `metadata.vw_LoadTestDAX` for a specific Track and Report.

In [ ]:
def fetch_test_queries(track: str, report: str, config_jdbc_url: str) -> DataFrame | None:
    """Query vw_LoadTestDAX for DAX query definitions."""
    sql = (
        "SELECT DISTINCT id, TrackName, ReportName, PageName, VisualID, VisualType, VisualTitle, "
        "DAXQuery as query "
        f"FROM metadata.vw_LoadTestDAX "
        f"WHERE trackName = '{track}' AND reportName = '{report}'"
    )
    try:
        df = readSQLData(sql, config_jdbc_url)
        if isinstance(df, list):
            df = spark.createDataFrame(df)
        return df
    except Exception as ex:
        print(f"[ERROR] fetch_test_queries failed: {ex}")
        return None

Executes all DAX queries page-by-page for both Import Mode (source) and Direct Lake Mode (target).
For each distinct report page in `config_df`, this function runs DAX queries sequentially  
against both environments and consolidates the results for performance comparison.

In [ ]:
def run_queries_for_pages(config_df: DataFrame, source_url: str, target_url: str):
    """
    For each distinct page in config_df:
      - filter config for that page
      - run queries sequentially in source & target
      - collect results
    Returns (src_final_df, tgt_final_df) or (None, None) on error.
    """
    try:
        page_rows = config_df.select("PageName").distinct().collect()
    except Exception as ex:
        print(f"[ERROR] Getting distinct pages failed: {ex}")
        return None, None

    all_src = []
    all_tgt = []

    for pr in page_rows:
        page = pr["PageName"]
        print(f"Processing page: {page}")
        try:
            df_page = config_df.filter(col("PageName") == page)
            query_list = df_page.collect()
        except Exception as ex:
            print(f"[WARN] filtering or collecting page '{page}' failed: {ex}")
            continue

        try:
            src_res = executeDAXqueries_sequentially(query_list, source_url, 50)
        except Exception as ex:
            print(f"[ERROR] Source queries run failed for page '{page}': {ex}")
            src_res = None

        try:
            tgt_res = executeDAXqueries_sequentially(query_list, target_url, 50)
        except Exception as ex:
            print(f"[ERROR] Target queries run failed for page '{page}': {ex}")
            tgt_res = None

        if src_res is not None:
            all_src.append(src_res)
        if tgt_res is not None:
            all_tgt.append(tgt_res)

    if not all_src or not all_tgt:
        print("[WARN] No valid results from queries.")
        return None, None

    try:
        src_final = reduce(DataFrame.unionAll, all_src)
        tgt_final = reduce(DataFrame.unionAll, all_tgt)
        return src_final, tgt_final
    except Exception as ex:
        print(f"[ERROR] Combining result DataFrames failed: {ex}")
        return None, None

Adds a `Status` column to both Import Mode (source) and Direct Lake Mode (target)
result DataFrames based on the text content of the `Result` column.

In [ ]:
def derive_status_columns(src_df: DataFrame, tgt_df: DataFrame):
    """Add a Status column to the result DataFrames based on the Result text."""
    try:
        df1 = src_df.withColumn(
            "Status",
            when(col("Result").contains("pbi.error"), "Failed").otherwise("Success")
        )
        df2 = tgt_df.withColumn(
            "Status",
            when(col("Result").contains("error"), "Failed").otherwise("Success")
        )
        return df1, df2
    except Exception as ex:
        print(f"[ERROR] derive_status_columns failed: {ex}")
        return None, None

Writes the final performance comparison results into the logging table **`logging.tbl_LoadTestDAX`** for tracking execution metrics.

In [ ]:
def log_and_persist_result(result_df: DataFrame, config_jdbc_url: str):
    """Persist the result DataFrame into the logging table."""
    if result_df is None:
        print("[WARN] No result DataFrame to write.")
        return
    try:
        writeSQLData(result_df, "logging.tbl_LoadTestDAX", config_jdbc_url)
    except Exception as ex:
        print(f"[ERROR] log_and_persist_result failed: {ex}")

Joins Import Mode (Power BI model) and Direct Lake Mode (Fabric model) results with the configuration DataFrame to produce a unified comparison result
To combine execution results for both environments, enrich them with metadata (Track, Report, Page, Visual), and attach a unique `RuntimeId` for traceability in performance logging.

In [ ]:
def assemble_result_df(runtime_id: str) -> DataFrame | None:
    """
    Join source and target results with config and produce the final result table.
    """
    try:
        sql = f"""
        SELECT '{runtime_id}' AS RuntimeId, A.Id,
               C.TrackName, C.ReportName, C.PageName,
               C.VisualID, C.VisualType, C.VisualTitle,
               A.Query,
               ROUND(A.`Execution Time`, 2) AS ImportMode_ExecutionTime,
               A.Status AS ImportMode_Status,
               A.`Start Time` AS ImportMode_StartTime,
               A.`End Time` AS ImportMode_EndTime,
               ROUND(B.`Execution Time`, 2) AS DirectLake_ExecutionTime,
               B.Status AS DirectLakeMode_Status,
               B.`Start Time` AS DirectLake_StartTime,
               B.`End Time` AS DirectLake_EndTime
        FROM PowerBIModel_sequential_results A
        LEFT JOIN FabricModel_sequential_results B
          ON A.Id = B.Id AND A.`Execution Type` = B.`Execution Type`
        LEFT JOIN config_df C ON A.id = C.id
        """
        return spark.sql(sql)
    except AnalysisException as ae:
        print(f"[SQL ERROR] assemble_result_df: {ae}")
        return None
    except Exception as ex:
        print(f"[ERROR] assemble_result_df failed: {ex}")
        return None

In [ ]:

spark = SparkSession.builder.getOrCreate()

def analyze_and_persist_query_filters(
    dfData,  # Spark DataFrame that includes columns: TrackName, ReportName, PageName, VisualID, VisualTitle, DAXQuery
    extract_udf,  # UDF that takes a DAXQuery string and returns an array / list of {TableName, ColumnName, FilterExpression}
    target_table: str,
    jdbc_url: str
):
    """
    For each row in dfData:
      - Convert to a one-row DataFrame
      - Apply the UDF to extract filter expressions
      - Explode the results
      - Drop duplicates
      - Persist results to SQL via writeSQLData

    Args:
      dfData: Spark DataFrame with required columns.
      extract_udf: A Spark UDF (or pandas UDF) that returns array of structs/dicts with fields:
                   TableName, ColumnName, FilterExpression.
      target_table: fully qualified SQL table name (e.g. "metadata.tbl_ReportVisualDAXQueryAnalyzer")
      jdbc_url: JDBC connection string for writeSQLData
    """
    # Validate input
    required_cols = ["TrackName", "ReportName", "PageName", "VisualID", "VisualTitle", "DAXQuery"]
    for c in required_cols:
        if c not in dfData.columns:
            raise ValueError(f"Input DataFrame missing required column: {c}")

    # Iterate through rows
    rows = dfData.collect()
    if not rows:
        print("[INFO] No rows in dfData; skipping query-filter extraction.")
        return

    for row in rows:
        try:
            row_dict = row.asDict()
        except Exception as e:
            print(f"[WARN] Could not convert row to dict: {row} — {e}")
            continue

        try:
            single_row_df = spark.createDataFrame([row_dict])
        except Exception as e:
            print(f"[ERROR] Failed to create single-row DF from {row_dict}: {e}")
            continue

        try:
            # Apply the UDF
            extracted_df = single_row_df.withColumn("extracted", extract_udf(single_row_df["DAXQuery"]))
        except Exception as e:
            print(f"[ERROR] UDF application failed for {row_dict}: {e}")
            continue

        try:
            final_df = (
                extracted_df
                .select(
                    "TrackName",
                    "ReportName",
                    "PageName",
                    "VisualID",
                    "VisualTitle",
                    explode("extracted").alias("extracted_row")
                )
                .select(
                    "TrackName",
                    "ReportName",
                    "PageName",
                    "VisualID",
                    "VisualTitle",
                    col("extracted_row.TableName").alias("TableName"),
                    col("extracted_row.ColumnName").alias("ColumnName"),
                    col("extracted_row.FilterExpression").alias("FilterExpression")
                )
                .dropDuplicates()
            )
        except Exception as e:
            print(f"[ERROR] Failed to build final_df for row {row_dict}: {e}")
            continue

        # Optionally display for debugging
        # display(final_df)

        try:
            writeSQLData(final_df, target_table, jdbc_url)
        except Exception as e:
            print(f"[ERROR] writeSQLData failed for {row_dict} into {target_table}: {e}")
            continue

    print("[INFO] Query filter extraction & persistence complete.")


To analyze DAX queries stored in metadata tables, extract referenced tables/columns/filters, and persist the results for downstream performance or lineage analysis

In [ ]:
# UDF to extract table, column, and filter expressions from DAX using only Python (no pandas)
def extract_table_column_filters_from_dax(daxQuery):
    """
    Extracts table, column, and filter expression information from a DAX query string.
    Returns a list of dicts with keys: TableName, ColumnName, FilterExpression
    """
    # Use raw strings for regex patterns and single backslash
    colPattern = r"'([A-Za-z0-9_ ]+)'\[([A-Za-z0-9_ ]+)\]"
    colRefs = re.findall(colPattern, daxQuery)
    filterPattern = r"('([A-Za-z0-9_ ]+)'\[([A-Za-z0-9_ ]+)\]\s*[<>=!]+[^,\n]*)"
    filterRefs = re.findall(filterPattern, daxQuery)
    filterMap = defaultdict(list)
    for fullExpr, table, column in filterRefs:
        filterMap[(table, column)].append(fullExpr.strip())
    rows = []
    for table, column in colRefs:
        filterExpr = '; '.join(filterMap[(table, column)]) if (table, column) in filterMap else ''
        rows.append({
            'TableName': table,
            'ColumnName': column,
            'FilterExpression': filterExpr
        })
    return rows

schema = ArrayType(StructType([
    StructField('TableName', StringType()),
    StructField('ColumnName', StringType()),
    StructField('FilterExpression', StringType())
]))

extract_udf = udf(extract_table_column_filters_from_dax, schema)

Scans a given Lakehouse directory for Power BI report JSON files and extracts their metadata identifiers (TrackName, ReportName, PageName) based on filename convention.

To dynamically discover uploaded JSON files and return their metadata for subsequent ingestion or DAX extraction.

In [ ]:
def list_json_files_with_metadata(directory: str) -> List[Tuple[str, str, str, str]]:
    """
    Return a list of tuples (full_path, trackName, reportName, pageName)
    for all JSON files under `directory`.
    Raises RuntimeError or ValueError on errors.
    """
    try:
        files = mssparkutils.fs.ls(directory)
    except Exception as e:
        raise RuntimeError(f"Cannot list files in directory '{directory}': {e}") from e

    results = []
    for f in files:
        if not hasattr(f, 'path'):
            # Skip entries without a path
            continue
        p = f.path
        if not p.lower().endswith(".json"):
            continue

        # Extract filename without extension
        basename = os.path.basename(p)
        name_no_ext = basename[:-len(".json")] if basename.lower().endswith(".json") else basename
        parts = name_no_ext.split("_")
        if len(parts) != 3:
            # Depending on policy, either skip or raise
            raise ValueError(f"JSON file '{p}' name doesn’t split into 3 parts by '_'.")
        trackName, reportName, pageName = parts
        results.append((p, trackName, reportName, pageName))

    return results

Retrieves all distinct `(TrackName, ReportName, PageName)` tuples from the `metadata.tbl_ReportVisualDAX` table to identify already-loaded reports.

Used to prevent re-inserting duplicate report metadata during ingestion or updates. 

In [ ]:
def load_existing_metadata(jdbc_url: str):
    """
    Query the metadata table and return a set of normalized (track, report, page) tuples.
    Handles both DataFrame and list/dict row types.
    """
    existing = set()
    try:
        sql = "SELECT distinct TrackName, ReportName, PageName FROM metadata.tbl_ReportVisualDAX"
        md_df = readSQLData(sql, jdbc_url)
    except Exception as e:
        print(f"[WARN] Could not query existing metadata: {e}")
        return existing

    # Determine how to iterate rows
    if hasattr(md_df, "collect"):
        try:
            rows = md_df.collect()
        except Exception as e:
            print(f"[WARN] Cannot collect from md_df: {e}")
            rows = []
    elif isinstance(md_df, list):
        rows = md_df
    else:
        print(f"[WARN] Unexpected md_df type: {type(md_df)}; treating as empty")
        rows = []

    for row in rows:
        try:
            # row could be dict, or Spark Row, or tuple/list
            if isinstance(row, dict):
                t = row.get("TrackName")
                r = row.get("ReportName")
                p = row.get("PageName")
            elif hasattr(row, "__getitem__") and not hasattr(row, "__dict__"):
                # like tuple or list
                t, r, p = row[0], row[1], row[2]
            else:
                # could be Row or object with attributes
                t = getattr(row, "TrackName", None)
                r = getattr(row, "ReportName", None)
                p = getattr(row, "PageName", None)

            # At this point, t, r, p might be None or non-string, so convert safely
            t_norm = str(t).strip().lower() if t is not None else ""
            r_norm = str(r).strip().lower() if r is not None else ""
            p_norm = str(p).strip().lower() if p is not None else ""

            existing.add((t_norm, r_norm, p_norm))
        except Exception as e:
            print(f"[WARN] Failed normalizing metadata row {row}: {e}")

    return existing

Parses a Power BI JSON telemetry file, flattens event structures, and extracts DAX query details (visualId, visualTitle, visualType, QueryText).  
To extract and standardize DAX queries from Power BI report JSONs for performance benchmarking.

In [ ]:
def extract_dax_from_json(input_path: str, track: str, report: str, page: str):
    """
    Load JSON from input_path, run Spark SQL to extract DAX query rows, return DataFrame.
    Raises on failure.
    """
    try:
        df = spark.read.option("multiline", "true").json(input_path)
        df.createOrReplaceTempView("vwRawDF")
    except Exception as e:
        raise RuntimeError(f"Failed reading JSON '{input_path}': {e}") from e

    sql = f"""
        WITH cteFlattenJSON AS (
            SELECT
                event.component,
                event.end,
                event.id,
                event.name,
                event.parentId,
                event.start,
                event.metrics.QueryText AS QueryText,
                event.metrics.RowCount AS RowCount,
                event.metrics.initialLoad AS initialLoad,
                event.metrics.sourceLabel AS sourceLabel,
                event.metrics.status AS status,
                event.metrics.visualId AS visualId,
                event.metrics.visualTitle AS visualTitle,
                event.metrics.visualType AS visualType
            FROM vwRawDF
            LATERAL VIEW explode(events) t AS event
        )
        SELECT
            '{track}' AS TrackName,
            '{report}' AS ReportName,
            '{page}' AS PageName,
            e.visualId,
            e.visualTitle,
            e.visualType,
            a.QueryText AS DAXQuery,
            true AS IsActive
        FROM cteFlattenJSON a
        JOIN cteFlattenJSON b ON a.parentId = b.id
        JOIN cteFlattenJSON c ON b.parentId = c.id
        JOIN cteFlattenJSON d ON c.parentId = d.id
        LEFT JOIN cteFlattenJSON e ON c.parentId = e.id
        WHERE a.QueryText IS NOT NULL
    """

    try:
        result_df = spark.sql(sql)
    except Exception as e:
        raise RuntimeError(f"Error executing SQL for JSON '{input_path}': {e}") from e

    return result_df

Checks each JSON file in the given directory, extracts DAX metadata, and inserts new entries into `metadata.tbl_ReportVisualDAX` only if they do not already exist.
Automates ingestion of Power BI DAX metadata from report JSON files, ensuring no duplicate `(TrackName, ReportName, PageName)` combinations.

In [ ]:
def upsert_metadata_from_jsons(json_directory: str, jdbc_url: str):
    """
    For each JSON file, check normalized existence, then insert new metadata.
    """
    try:
        file_tuples = list_json_files_with_metadata(json_directory)
    except Exception as e:
        print(f"[ERROR] Could not list JSON files in '{json_directory}': {e}")
        return

    existing = load_existing_metadata(jdbc_url)
    to_insert = []

    for (path, track, report, page) in file_tuples:
        key_norm = (track.strip().lower(), report.strip().lower(), page.strip().lower())
        print("Checking key:", key_norm, "exists?", key_norm in existing)

        if key_norm in existing:
            print(f"[SKIP] Metadata already exists for {track}, {report}, {page}")
        else:
            print(f"[NEW] Will insert metadata for {track}, {report}, {page}")
            try:
                df = extract_dax_from_json(path, track, report, page)
                # df is expected to be Spark DataFrame; we collect rows
                for row in df.collect():
                    to_insert.append({
                        "TrackName": track,
                        "ReportName": report,
                        "PageName": page,
                        "visualId": getattr(row, "visualId", None),
                        "visualTitle": getattr(row, "visualTitle", None),
                        "visualType": getattr(row, "visualType", None),
                        "DAXQuery": getattr(row, "DAXQuery", None),
                        "IsActive": True
                    })
            except Exception as e:
                print(f"[ERROR] Failed to extract DAX from {path}: {e}")

    if to_insert:
        insert_metadata_rows(to_insert, jdbc_url)
    else:
        print("[INFO] No new metadata to insert.")

Inserts extracted DAX metadata rows into the `metadata.tbl_ReportVisualDAX` table. 
Safely persist newly discovered DAX queries (from Power BI report JSONs) into the metadata table, avoiding SQL injection or malformed inserts.

In [ ]:
def insert_metadata_rows(rows: list[dict], jdbc_url: str):
    """
    Insert each metadata row into the table. Uses your executeSQL or writeSQLData function.
    """
    for r in rows:
        # Build an INSERT. Use proper escaping for single quotes.
        track = r["TrackName"].replace("'", "''")
        report = r["ReportName"].replace("'", "''")
        page = r["PageName"].replace("'", "''")
        vid = (r.get("visualId") or "").replace("'", "''")
        vtitle = (r.get("visualTitle") or "").replace("'", "''")
        vtype = (r.get("visualType") or "").replace("'", "''")
        dax = (r.get("DAXQuery") or "").replace("'", "''")
        is_active = 1 if r.get("IsActive", True) else 0

        sql = f"""
        INSERT INTO metadata.tbl_ReportVisualDAX
          (TrackName, ReportName, PageName, visualId, visualTitle, visualType, DAXQuery, IsActive)
        VALUES
          ('{track}', '{report}', '{page}', '{vid}', '{vtitle}', '{vtype}', '{dax}', {is_active})
        """
        try:
            # Use your execution method (or JDBC)
            _ = readSQLData(sql, jdbc_url)
            print(f"[INFO] Inserted metadata: {track}, {report}, {page}")
        except Exception as ie:
            print(f"[ERROR] Insert failed for {track}, {report}, {page}: {ie}")

Retrieves distinct `(TrackName, ReportName, PageName)` tuples from the **metadata.tbl_ReportVisualDAXQueryAnalyzer** table.
Prevents duplicate inserts when extracting and persisting DAX query-analyzer results from Power BI reports.  
Normalizes keys to lowercase and trims whitespace for reliable comparisons.

In [ ]:
def load_existing_rvqa_keys(jdbc_url: str):
    """
    Load existing keys from tbl_ReportVisualDAXQueryAnalyzer as a normalized set.
    Returns set of (track, report, page) lowercased/stripped.
    """
    existing = set()
    try:
        sql = "SELECT distinct TrackName, ReportName, PageName FROM metadata.tbl_ReportVisualDAXQueryAnalyzer"
        md_df = readSQLData(sql, jdbc_url)
    except Exception as e:
        print(f"[WARN] Could not query existing query-analyzer table: {e}")
        return existing

    # Handle list or DataFrame
    if hasattr(md_df, "collect"):
        rows = md_df.collect()
    elif isinstance(md_df, list):
        rows = md_df
    else:
        print(f"[WARN] Unexpected type from readSQLData: {type(md_df)}")
        rows = []

    for row in rows:
        try:
            if isinstance(row, dict):
                t = row.get("TrackName")
                r = row.get("ReportName")
                p = row.get("PageName")
            elif hasattr(row, "__getitem__") and not hasattr(row, "__dict__"):
                t, r, p = row[0], row[1], row[2]
            else:
                t = getattr(row, "TrackName", None)
                r = getattr(row, "ReportName", None)
                p = getattr(row, "PageName", None)

            tn = str(t).strip().lower() if t is not None else ""
            rn = str(r).strip().lower() if r is not None else ""
            pn = str(p).strip().lower() if p is not None else ""
            existing.add((tn, rn, pn))
        except Exception as ex:
            print(f"[WARN] Could not normalize existing analyzer row {row}: {ex}")
    return existing

Extracts filter components from DAX queries and inserts them into **metadata.tbl_ReportVisualDAXQueryAnalyzer**, skipping duplicates that already exist.

To avoid reprocessing Power BI reports whose (TrackName, ReportName, PageName) combinations already exist in the query analyzer table.

In [ ]:
def analyze_and_persist_query_filters_if_new(
    dfData,
    extract_udf,
    target_table: str,
    jdbc_url: str
):
    """
    Similar to analyze_and_persist_query_filters, but only inserts rows
    where (TrackName,ReportName,PageName) combo is not already in
    tbl_ReportVisualDAXQueryAnalyzer.
    """
    # Load existing keys for analyzer table
    existing_keys = load_existing_rvqa_keys(jdbc_url)

    required = ["TrackName", "ReportName", "PageName", "DAXQuery", "VisualID", "VisualTitle"]
    for c in required:
        if c not in dfData.columns:
            raise ValueError(f"dfData is missing required column: {c}")

    rows = dfData.collect()
    if not rows:
        print("[INFO] No rows to process for query filters.")
        return

    for row in rows:
        # Normalize the key
        t = getattr(row, "TrackName", None)
        r = getattr(row, "ReportName", None)
        p = getattr(row, "PageName", None)
        key_norm = (str(t).strip().lower(), str(r).strip().lower(), str(p).strip().lower())

        if key_norm in existing_keys:
            print(f"[SKIP] Analyzer already has entry for {t}, {r}, {p}")
            continue

        # If not existing, proceed to extract & insert
        try:
            single_row_df = spark.createDataFrame([row.asDict()])
            extracted = single_row_df.withColumn("extracted", extract_udf(single_row_df["DAXQuery"]))
            final_df = (
                extracted
                .select("TrackName","ReportName","PageName","VisualID","VisualTitle", explode("extracted").alias("extracted_row"))
                .select(
                    "TrackName","ReportName","PageName","VisualID","VisualTitle",
                    col("extracted_row.TableName").alias("TableName"),
                    col("extracted_row.ColumnName").alias("ColumnName"),
                    col("extracted_row.FilterExpression").alias("FilterExpression")
                )
                .dropDuplicates()
            )
            writeSQLData(final_df, target_table, jdbc_url)
            print(f"[INSERT] Analyzer data for {t}, {r}, {p}")
        except Exception as e:
            print(f"[ERROR] During filter extraction or insert for {t}, {r}, {p}: {e}")
            continue

Orchestrates the process of extracting and persisting DAX query filters into **metadata.tbl_ReportVisualDAXQueryAnalyzer**, skipping duplicates.

To refresh or initialize the Query Analyzer metadata table based on the latest entries in **metadata.tbl_ReportVisualDAX** (which stores DAX queries extracted from Power BI reports).

In [ ]:
def run_query_analyzer_metadata_entry(config_jdbc_url: str, extract_udf, target_table: str = "metadata.tbl_ReportVisualDAXQueryAnalyzer"):
    """
    Fetches distinct entries from tbl_ReportVisualDAX, and for those not already in the
    analyzer table, extracts filter info and inserts them.
    """
    try:
        config_query = """
            SELECT DISTINCT TrackName, ReportName, PageName, VisualID, VisualTitle, DAXQuery
            FROM metadata.tbl_ReportVisualDAX
        """
        dfData = readSQLData(config_query, config_jdbc_url)
    except Exception as e:
        print(f"[ERROR] Failed to read base metadata from tbl_ReportVisualDAX: {e}")
        return

    # If readSQLData returns a list, convert to Spark DataFrame
    if isinstance(dfData, list):
        try:
            dfData = spark.createDataFrame(dfData)
        except Exception as e:
            print(f"[ERROR] Failed to convert list to Spark DataFrame: {e}")
            return

    try:
        analyze_and_persist_query_filters_if_new(
            dfData,
            extract_udf = extract_udf,
            target_table = target_table,
            jdbc_url = config_jdbc_url
        )
    except Exception as e:
        print(f"[ERROR] analyze_and_persist_query_filters_if_new failed: {e}")
        return

    print("[INFO] run_query_analyzer_update completed.")


Executes a metadata SQL deployment script on a specified SQL Server database.

Automates the creation or alteration of metadata objects (e.g., tables, views) by reading and executing a `.sql` file against the target Fabric or SQL environment.

In [ ]:
def run_metadata_deployment(server: str, database: str, sql_script_path: str):
    """
    Builds the JDBC URL, runs the metadata script, and prints a summary with error handling.
    """
    try:
        config_jdbc_url = (
            "jdbc:sqlserver://{};databaseName={};"
            "trustServerCertificate=false;loginTimeout=30;"
        ).format(server, database)
    except Exception as e:
        print(f"Error formatting JDBC URL for server={server}, database={database} : {e}")
        return   # or raise, depending on how critical this is

    try:
        exec_results = execute_sql_metadata_script(sql_script_path, config_jdbc_url)
    except Exception as e:
        print(f"Error executing metadata SQL script '{sql_script_path}' : {e}")
        return

    print("Execution summary:")
    for r in exec_results:
        status = r.get("status", "no-status")
        error = r.get("error")
        print(f"{status} - {error or 'OK'}")

Executes an ad-hoc SQL query via JDBC using the Spark JVM bridge and Azure AD token authentication.

Provides a reusable, secure way to fetch data from Azure SQL Database or Fabric SQL endpoints within the Performance Tool framework, using Managed Identity or Power BI service principal token.

In [ ]:
def readSQLData(query, jdbc_url, access_token=mssparkutils.credentials.getToken("pbi")):
    try:
        # Create a spark properties object and pass the access token
        properties = spark._sc._gateway.jvm.java.util.Properties()
        properties.setProperty("accessToken", access_token)
        properties.setProperty("encrypt", "true")
        driver_manager = spark._sc._gateway.jvm.java.sql.DriverManager

        # Create a connection object and pass the properties object
        con = driver_manager.getConnection(jdbc_url, properties)

        # Create callable statement and execute it
        exec_statement = con.prepareCall(query)
        start_time = datetime.now()
        exec_statement.execute()
        end_time = datetime.now()
        result_set = exec_statement.getResultSet()
        data = []
        if result_set:
            while result_set.next():
                row = {}
                for i in range(1, result_set.getMetaData().getColumnCount() + 1):
                    row[result_set.getMetaData().getColumnName(i)] = result_set.getString(i)
                data.append(row)
        exec_statement.close()
        con.close()

        # Return the execution time and timestamps
        return data
    except Exception as e:
        print(f"Error: {e}")
        return None

StatementMeta(, 8eb609f1-4cac-462d-a95e-6e6e1fd9366c, 5, Finished, Available, Finished)

Writes a Spark or Pandas DataFrame into a SQL table using Azure AD token-based authentication.

To provide a secure and reusable method to insert or append records into SQL tables (Fabric Warehouse, Lakehouse SQL Endpoint, or Azure SQL Database).

In [ ]:
def writeSQLData(df, table_name: str, jdbc_url: str, access_token: str = None):
    """
    Appends the DataFrame `df` into the SQL table `table_name` via JDBC using a token for auth.
    Returns True if success, False or raises exception otherwise.
    """
    try:
        # If no access_token passed, fetch default
        if access_token is None:
            access_token = mssparkutils.credentials.getToken("pbi")

        # Convert pandas.DataFrame to Spark DataFrame if needed
        if isinstance(df, pd.DataFrame):
            spark_df = spark.createDataFrame(df)
        else:
            spark_df = df

        # Basic validation
        if spark_df.rdd.isEmpty():
            print(f"[INFO] DataFrame is empty. Nothing to write to {table_name}.")
            return True  # no error, but no write

        # Write to SQL via JDBC
        (spark_df.write
            .format("jdbc")
            .option("url", jdbc_url)
            .option("dbtable", table_name)
            .option("accessToken", access_token)
            .option("encrypt", "true")
            .mode("append")
            .save()
        )

        return True

    except Exception as e:
        print(f"[ERROR] writeSQLData failed for table {table_name}: {e}")
        return False

In [ ]:
#Escapes single quotes in SQL string literals to prevent syntax errors or SQL injection issues.
def _escape_sql_literal(val: str) -> str:
    """Escape single quotes within SQL string literals by doubling them."""
    if val is None:
        return ""
    return str(val).replace("'", "''")

Inserts multiple connection metadata rows into `metadata.tbl_LoadTestConnectionDetailsDAX`.
Automates population of DAX performance tool configuration entries for Import Mode and Direct Lake Mode datasets, avoiding manual SQL edits.
###### **Highlights:**
###### - Validates required keys
###### - Escapes single quotes safely using `_escape_sql_literal`
###### - Converts to Spark DataFrame for bulk append via `writeSQLData`
###### - Logs actions clearly for debugging in Fabric Notebooks

In [ ]:
def insert_multiple_connection_details(conn_details_list: list[dict], jdbc_url: str):
    """
    Insert multiple rows into metadata.tbl_LoadTestConnectionDetailsDAX.
    conn_details_list: list of dicts, each dict has the same keys as your single insert.
    Returns True/False depending on success.
    """
    if not conn_details_list:
        print("[INFO] No entries to insert.")
        return True

    # Validate each dict has required keys
    required = [
        "reportName",
        "importModeWorkspaceName",
        "importModeWorkspaceId",
        "importModeDatasetName",
        "importModeDatasetId",
        "directLakeModeWorkspaceName",
        "directLakeModeWorkspaceId",
        "directLakeModeDatasetName",
        "directLakeModeDatasetId",
        "isActive"
    ]
    for i, d in enumerate(conn_details_list):
        missing = [k for k in required if k not in d]
        if missing:
            raise ValueError(f"Entry at index {i} is missing required keys: {missing}")

    # Clean / normalize entries (e.g. cast isActive to int 0/1, escape strings if needed)
    cleaned = []
    for d in conn_details_list:
        row = {
            "reportName": str(d["reportName"]) if d["reportName"] is not None else None,
            "importModeWorkspaceName": str(d["importModeWorkspaceName"]) if d["importModeWorkspaceName"] is not None else None,
            "importModeWorkspaceId": str(d["importModeWorkspaceId"]) if d["importModeWorkspaceId"] is not None else None,
            "importModeDatasetName": str(d["importModeDatasetName"]) if d["importModeDatasetName"] is not None else None,
            "importModeDatasetId": str(d["importModeDatasetId"]) if d["importModeDatasetId"] is not None else None,
            "directLakeModeWorkspaceName": str(d["directLakeModeWorkspaceName"]) if d["directLakeModeWorkspaceName"] is not None else None,
            "directLakeModeWorkspaceId": str(d["directLakeModeWorkspaceId"]) if d["directLakeModeWorkspaceId"] is not None else None,
            "directLakeModeDatasetName": str(d["directLakeModeDatasetName"]) if d["directLakeModeDatasetName"] is not None else None,
            "directLakeModeDatasetId": str(d["directLakeModeDatasetId"]) if d["directLakeModeDatasetId"] is not None else None,
            "isActive": 1 if d["isActive"] else 0
        }
        cleaned.append(row)

    try:
        # Create a Spark DataFrame from the list of rows
        df_to_insert = spark.createDataFrame(cleaned)
    except Exception as e:
        print(f"[ERROR] Could not build DataFrame for multiple inserts: {e}")
        return False

    # Use your existing writeSQLData to append all rows
    success = writeSQLData(df_to_insert, "metadata.tbl_LoadTestConnectionDetailsDAX", jdbc_url)
    return success


Generates a parameterized SQL `INSERT ... WHERE NOT EXISTS` statement for inserting connection metadata into `metadata.tbl_LoadTestConnectionDetailsDAX`.

Prevents duplicate entries for the same `reportName` while allowing idempotent metadata inserts (especially useful in automated deployment scripts and connection setup utilities).

In [ ]:
def build_insert_if_not_exists_sql(conn: dict) -> str:
    """
    Given a connection-details dict, build a SQL INSERT … WHERE NOT EXISTS statement.
    """
    required = [
        "reportName",
        "importModeWorkspaceName", "importModeWorkspaceId",
        "importModeDatasetName", "importModeDatasetId",
        "directLakeModeWorkspaceName", "directLakeModeWorkspaceId",
        "directLakeModeDatasetName", "directLakeModeDatasetId",
        "isActive"
    ]
    missing = [k for k in required if k not in conn]
    if missing:
        raise ValueError(f"Missing required keys: {missing}")

    # Escape each string field
    rn = _escape_sql_literal(conn["reportName"])
    im_wsn = _escape_sql_literal(conn["importModeWorkspaceName"])
    im_wsid = _escape_sql_literal(conn["importModeWorkspaceId"])
    im_dsn = _escape_sql_literal(conn["importModeDatasetName"])
    im_dsid = _escape_sql_literal(conn["importModeDatasetId"])
    dl_wsn = _escape_sql_literal(conn["directLakeModeWorkspaceName"])
    dl_wsid = _escape_sql_literal(conn["directLakeModeWorkspaceId"])
    dl_dsn = _escape_sql_literal(conn["directLakeModeDatasetName"])
    dl_dsid = _escape_sql_literal(conn["directLakeModeDatasetId"])
    is_act = 1 if conn["isActive"] else 0

    sql = f"""
    INSERT INTO metadata.tbl_LoadTestConnectionDetailsDAX
        (reportName,
         importModeWorkspaceName,
         importModeWorkspaceId,
         importModeDatasetName,
         importModeDatasetId,
         directLakeModeWorkspaceName,
         directLakeModeWorkspaceId,
         directLakeModeDatasetName,
         directLakeModeDatasetId,
         isActive)
    SELECT
        '{rn}' AS reportName,
        '{im_wsn}' AS importModeWorkspaceName,
        '{im_wsid}' AS importModeWorkspaceId,
        '{im_dsn}' AS importModeDatasetName,
        '{im_dsid}' AS importModeDatasetId,
        '{dl_wsn}' AS directLakeModeWorkspaceName,
        '{dl_wsid}' AS directLakeModeWorkspaceId,
        '{dl_dsn}' AS directLakeModeDatasetName,
        '{dl_dsid}' AS directLakeModeDatasetId,
        {is_act} AS isActive
    WHERE NOT EXISTS (
        SELECT 1 FROM metadata.tbl_LoadTestConnectionDetailsDAX AS t
        WHERE LOWER(t.reportName) = LOWER('{rn}')
    );
    """
    return sql

Batch executor for inserting multiple connection configuration records into `metadata.tbl_LoadTestConnectionDetailsDAX` using safe “INSERT IF NOT EXISTS” logic.

Loops through a list of connection detail dictionaries, builds SQL using
`build_insert_if_not_exists_sql()`, and executes each insert via a provided execution function.

In [ ]:
def batch_insert_conn_if_not_exists(conn_list: list[dict], jdbc_url: str, exec_fn) -> list[dict]:
    """
    Loop over a list of connection dicts, build & execute SQL with “NOT EXISTS” logic.
    exec_fn(sql: str) => executes the SQL (e.g. via JDBC or your token-based execution).
    Returns a list of result dicts for each attempted insert.
    """
    results = []
    for conn in conn_list:
        try:
            sql = build_insert_if_not_exists_sql(conn)
        except Exception as e:
            results.append({
                "conn": conn,
                "status": "failed",
                "error": f"SQL build error: {e}"
            })
            continue

        try:
            resp = exec_fn(sql)
            results.append({
                "conn": conn,
                "status": "success",
                "response": resp
            })
        except Exception as ex:
            results.append({
                "conn": conn,
                "status": "failed",
                "error": str(ex),
                "sql": sql
            })
    return results

Executes one or more raw SQL statements against a Fabric or Azure SQL Database
using **Managed Identity / Service Principal token authentication**.

Provides a reusable low-level executor for running DDL/DML statements (used by batch insertion, schema creation, or metadata deployment utilities).

In [ ]:
def executeSQLStatementsWithToken(sql_statements: list[str], jdbc_url: str, access_token: str = None):
    """
    Execute one or more raw T-SQL statements via JDBC, using an access token.
    Returns list of dicts for each statement with status / error.
    """
    if access_token is None:
        access_token = mssparkutils.credentials.getToken("pbi")

    props = spark._sc._gateway.jvm.java.util.Properties()
    props.setProperty("accessToken", access_token)
    props.setProperty("encrypt", "true")

    driver_manager = spark._sc._gateway.jvm.java.sql.DriverManager
    conn = driver_manager.getConnection(jdbc_url, props)

    results = []
    for sql in sql_statements:
        try:
            stmt = conn.createStatement()
            stmt.execute(sql)
            results.append({"sql": sql, "status": "success", "error": None})
            stmt.close()
        except Exception as ex:
            results.append({"sql": sql, "status": "failed", "error": str(ex)})
    conn.close()
    return results


Executes a DAX query against a Power BI dataset (Import or Direct Lake) using the REST API.  

Sends the provided DAX query to the `/executeQueries` endpoint for a given dataset, returning only the raw JSON response or an error indicator.

In [ ]:
def executeDAXqueries_resultonly(query, url):
    try:
        # Headers
        headers = {
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {mssparkutils.credentials.getToken("pbi")}'
        }

        payload = {
            "queries": [
                {
                    "query": query
                }
            ]
        }
        start_time = datetime.now()
        response = requests.post(url, json=payload, headers=headers)
        end_time = datetime.now()
        duration = round((end_time - start_time).total_seconds(), 2)
        if response.status_code == 200:
            return response.json()
        else:
            return -1
    except Exception as e:
        print(f"Error: {e}")
        return None

StatementMeta(, 8eb609f1-4cac-462d-a95e-6e6e1fd9366c, 7, Finished, Available, Finished)

Executes a DAX query against a Power BI dataset (Import or Direct Lake) using the REST API, and captures timing, status, and full response payload for performance comparison.

Core function used in the Performance Tool for executing DAX queries sequentially (via Import Mode and Direct Lake datasets) and benchmarking their runtime and response validity.

In [ ]:
def executeDAXqueries(config, url):
    try:
        # Headers
        headers = {
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {mssparkutils.credentials.getToken("pbi")}'
        }

        payload = {
            "queries": [
                {
                    "query": config.query
                }
            ]
        }
        start_time = datetime.now()
        response = requests.post(url, json=payload, headers=headers)
        end_time = datetime.now()
        duration = round((end_time - start_time).total_seconds(), 2)
        if response.status_code == 200:
            return config.id, config.query, start_time, end_time, duration, str(response.json())
        else:
            return config.id, config.query, start_time, end_time, duration, f"Error: {response.status_code} - {response.text}"
    except Exception as e:
        print(f"Error: {e}")
        return config.id, config.query, None, None, None, f"Exception: {e}"

StatementMeta(, 8eb609f1-4cac-462d-a95e-6e6e1fd9366c, 8, Finished, Available, Finished)

Executes multiple DAX queries in **parallel threads** against a Power BI dataset using the `/executeQueries` REST API endpoint.

Enables concurrent query benchmarking across Import Mode and Direct Lake datasets to analyze scalability, performance, and parallel execution characteristics.

In [ ]:
# Parallel execution
def executeDAXqueries_concurrently(queries, url, concurrency):
    try:
        results = []
        total_start_time = datetime.now()

        with concurrent.futures.ThreadPoolExecutor(max_workers=int(concurrency)) as executor:
            future_to_query = {executor.submit(executeDAXqueries, query, url): query for query in queries}
            for future in concurrent.futures.as_completed(future_to_query):
                id, query, start_time, end_time, duration, result = future.result()
                results.append((id, query, start_time, end_time, duration, result))

        total_end_time = datetime.now()
        total_execution_time = round((total_end_time - total_start_time).total_seconds(), 2)

        # Create a Spark DataFrame for logging
        df = spark.createDataFrame(results, schema=schema)
        df = df.withColumn("Total Execution Time", lit(total_execution_time))
        df = df.withColumn("Execution Type", lit('CON'))
        return df
    except Exception as e:
        print(f"Error: {e}")
        return None

StatementMeta(, 8eb609f1-4cac-462d-a95e-6e6e1fd9366c, 9, Finished, Available, Finished)

Executes a list of DAX queries sequentially (or concurrently if max_workers > 1) against a Power BI dataset via the `/executeQueries` REST API.

Core function used in Import Mode and Direct Lake benchmarking workflows to execute all visual DAX queries either sequentially or with mild concurrency.

In [ ]:
def executeDAXqueries_sequentially(queries, url, max_workers=1):
    """
    Executes all DAX queries concurrently (with max_workers threads, defaults to 1).
    When max_workers > 1, it behaves like a concurrent execution; when 1, it's purely sequential.
    """
    try:
        results = []
        total_start_time = datetime.now()

        with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
            future_to_query = {executor.submit(executeDAXqueries, query, url): query for query in queries}
            for future in concurrent.futures.as_completed(future_to_query):
                id, query, start_time, end_time, duration, result = future.result()
                results.append((id, query, start_time, end_time, duration, result))

        total_end_time = datetime.now()
        total_execution_time = round((total_end_time - total_start_time).total_seconds(), 2)

        # Create a Spark DataFrame for logging
        df = spark.createDataFrame(results, schema=schemaMain)
        df = df.withColumn("Total Execution Time", lit(total_execution_time))
        df = df.withColumn("Execution Type", lit('SEQ'))  # Use 'CON' if you want to mark it as concurrent
        return df
    except Exception as e:
        print(f"Error: {e}")
        return None

StatementMeta(, 8eb609f1-4cac-462d-a95e-6e6e1fd9366c, 10, Finished, Available, Finished)

Reads a `.sql` script stored under the **Lakehouse Files** area and splits it into individual T-SQL batches separated by `GO` statements (case-insensitive).

Used by Fabric deployment automation scripts (e.g., Metadata deployment, Connection Details setup) to dynamically execute multi-batch SQL files from `/Files/...` paths within the Lakehouse.

In [ ]:
def read_file_from_lakehouse(sql_path: str, max_bytes: int = 10_000_000) -> list[str]:
    """
    Read the .sql file from lakehouse / Files area, then split into T-SQL batches (GO).
    Returns a list of cleaned SQL batch strings.

    Raises:
      RuntimeError on failure (with chained original exception).
    """
    content = None
    try:
        content = mssparkutils.fs.head(sql_path, max_bytes)
        if content is None:
            raise IOError(f"No content returned when reading {sql_path}")
    except Exception as e:
        # Wrap and rethrow with context, preserving traceback
        raise RuntimeError(f"Failed to read SQL file from path '{sql_path}'") from e

    # Now split by GO batches
    try:
        # Normalize line endings
        text = content.replace("\r\n", "\n").replace("\r", "\n")
        # Split on lines that are just "GO" (case-insensitive)
        batches = re.split(r'(?im)^\s*GO\s*$', text)
        cleaned = [b.strip() for b in batches if b.strip()]
        return cleaned
    except re.error as rex:
        # Regex parsing failure
        raise RuntimeError(f"Error splitting T-SQL batches in file '{sql_path}'") from rex
    except Exception as ex:
        # Catch-all for unexpected errors during splitting
        raise RuntimeError(f"Unexpected error processing SQL script '{sql_path}'") from ex

StatementMeta(, 8eb609f1-4cac-462d-a95e-6e6e1fd9366c, 11, Finished, Available, Finished)

Splits a block of SQL text into individual executable statements.

Used in metadata deployment and automation pipelines to handle SQL scripts that contain multiple statements separated by semicolons (`;`), while safely ignoring
semicolons within string literals.

In [ ]:
def split_sql_statements(sql_text: str) -> list[str]:
    """
    Split the SQL text into individual statements (naive approach).
    Raises if input is invalid.
    """
    if not isinstance(sql_text, str):
        raise ValueError("sql_text must be a string")
    try:
        parts = re.split(r';\s*(?=(?:[^\'"]|\'[^\']*\'|"[^"]*")*$)', sql_text)
        stmts = [p.strip() for p in parts if p.strip()]
        return stmts
    except re.error as rex:
        raise RuntimeError("Regex failed splitting SQL statements") from rex

StatementMeta(, 8eb609f1-4cac-462d-a95e-6e6e1fd9366c, 12, Finished, Available, Finished)

In [ ]:
def execute_sql_statements(statements: list[str],jdbc_url: str,access_token: str = None) -> list[dict]:
    """
    Execute each SQL statement via JDBC using token authentication.
    Returns list of dicts with status / timing / error.
    """
    if access_token is None:
        access_token = mssparkutils.credentials.getToken("pbi")

    conn = None
    stmt = None
    results = []
    try:
        # Prepare Java Properties for JDBC
        props = spark._sc._gateway.jvm.java.util.Properties()
        props.setProperty("accessToken", access_token)
        props.setProperty("encrypt", "true")

        driver_manager = spark._sc._gateway.jvm.java.sql.DriverManager
        conn = driver_manager.getConnection(jdbc_url, props)
        stmt = conn.createStatement()

        for sql in statements:
            try:
                print("Executing SQL:", sql)
                ts_start = datetime.utcnow()
                stmt.execute(sql)
                ts_end = datetime.utcnow()
                results.append({
                    "sql": sql,
                    "status": "success",
                    "start": ts_start,
                    "end": ts_end,
                    "error": None
                })
            except Exception as ex_stmt:
                print("Error executing statement:", sql, "\nException:", ex_stmt)
                results.append({
                    "sql": sql,
                    "status": "failed",
                    "start": None,
                    "end": None,
                    "error": str(ex_stmt)
                })
                # continue to next statement rather than break
    except Exception as ex_conn:
        # If connection or statement creation fails entirely, wrap error
        raise RuntimeError(f"Failed to open JDBC connection or create statement: {ex_conn}") from ex_conn
    finally:
        # always attempt cleanup
        try:
            if stmt:
                stmt.close()
        except Exception as close_ex:
            print("Warning: failed to close statement:", close_ex)
        try:
            if conn:
                conn.close()
        except Exception as close_ex2:
            print("Warning: failed to close connection:", close_ex2)

    return results

StatementMeta(, 8eb609f1-4cac-462d-a95e-6e6e1fd9366c, 13, Finished, Available, Finished)

Reads a `.sql` file from the Lakehouse `Files/` area, splits it into executable T-SQL batches (using `GO` as delimiter), and executes each batch sequentially via JDBC using Managed Identity or Service Principal token.

Central driver for deploying metadata artifacts (tables, views, procedures, etc.)
into Fabric / Synapse SQL endpoints directly from automation notebooks.

In [ ]:
def execute_sql_metadata_script(sql_path: str,jdbc_url: str,access_token: str = None) -> list[dict]:
    """
    Read the .sql file, split into T-SQL batches (by GO), and execute them.
    Returns execution results.
    """
    try:
        sql_text = read_file_from_lakehouse(sql_path)
    except Exception as e:
        # If we cannot read the file, nothing more to do
        print("ERROR: cannot load SQL script:", e)
        # Optionally rethrow or return empty list
        raise  

    # Execute the batches
    return execute_sql_statements(sql_text, jdbc_url, access_token)

StatementMeta(, 8eb609f1-4cac-462d-a95e-6e6e1fd9366c, 14, Finished, Available, Finished)